In [ ]:
!nvidia-smi
!pip -q install -U datasets transformers accelerate "optimum[onnxruntime]" scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datasets import load_dataset
ds_all = load_dataset("rishia2220/icfd-31k")
print(ds_all)

In [ ]:
import pandas as pd
cols = ["conversation_uid","chunk_timestamp","final_verdict","case_type","domain"]
meta = {k: v.select_columns(cols).to_pandas() for k, v in ds_all.items()}
for k, df in meta.items():
    print(k, len(df), df.conversation_uid.nunique())
    print(pd.crosstab(df.case_type, df.final_verdict))
tr = set(meta["train"].conversation_uid)
va = set(meta["validation"].conversation_uid)
te = set(meta["test"].conversation_uid)
print("overlap train/val, train/test, val/test:", len(tr&va), len(tr&te), len(va&te))

In [ ]:
import numpy as np
def pick(df, per_conv):
    df = df.copy()
    df["idx"] = np.arange(len(df))
    df["frac"] = df.chunk_timestamp / df.groupby("conversation_uid").chunk_timestamp.transform("max")
    df["y"] = (df.final_verdict == "YES").astype(int)
    keep = df[(df.y == 0) | (df.frac >= 0.4)]
    return keep.groupby("conversation_uid", group_keys=False).apply(
        lambda g: g.sample(min(per_conv, len(g)), random_state=42))
sel = {"train": pick(meta["train"], 3),
       "validation": pick(meta["validation"], 2),
       "test": pick(meta["test"], 2)}
for k, v in sel.items():
    print(k, len(v), v.y.mean())

In [ ]:
import re
from datasets import Dataset
from transformers import AutoTokenizer
name = "distilbert-base-multilingual-cased"
tok = AutoTokenizer.from_pretrained(name, truncation_side="left")
def build(split):
    sub = ds_all[split].select(sel[split].idx.tolist())
    texts = [re.sub(r"\b(Agent|Customer):\s*", "", t) for t in sub["cumulative_text"]]
    d = Dataset.from_dict({"text": texts, "label": sel[split].y.tolist()})
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=256),
                 batched=True, remove_columns=["text"])
tr_ds, va_ds, te_ds = build("train"), build("validation"), build("test")
print(len(tr_ds), len(va_ds), len(te_ds))

In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, accuracy_score
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)
def metrics(p):
    pred = p.predictions.argmax(-1)
    return {"acc": accuracy_score(p.label_ids, pred), "f1": f1_score(p.label_ids, pred)}
args = TrainingArguments(output_dir="/content/out", num_train_epochs=2,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=3e-5, fp16=True, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
    data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

In [ ]:
out = trainer.predict(te_ds)
pred = out.predictions.argmax(-1)
t = sel["test"].copy()
t["pred"] = pred
print("overall acc/f1:", accuracy_score(t.y, pred), f1_score(t.y, pred))
print(t.groupby(["case_type", "y"]).pred.agg(["mean", "count"]))

In [ ]:
import torch, os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
m = AutoModelForSequenceClassification.from_pretrained("/content/model").to("cpu").eval()
m.config.return_dict = False
tok = AutoTokenizer.from_pretrained("/content/model")
os.makedirs("/content/onnx", exist_ok=True)
e = tok("hello there", return_tensors="pt")
torch.onnx.export(
    m, (e["input_ids"], e["attention_mask"]), "/content/onnx/model.onnx",
    input_names=["input_ids", "attention_mask"], output_names=["logits"],
    dynamic_axes={"input_ids": {0: "b", 1: "s"},
                  "attention_mask": {0: "b", 1: "s"},
                  "logits": {0: "b"}},
    opset_version=17, dynamo=False)
tok.save_pretrained("/content/onnx")
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic("/content/onnx/model.onnx", "/content/onnx/model_int8.onnx",
                 weight_type=QuantType.QInt8)
!ls -lh /content/onnx

In [ ]:
import re, torch
S = [
  ('c-s1', 'scam', 'dev', False, 'Good afternoon, I am calling from the fraud prevention team. We have detected suspicious activity on your debit card ending four four one nine. Two unauthorised charges were flagged less than ten minutes ago. I need to verify your identity before I can stop them. I am sending a secure authorisation token to your phone right now, please read that six digit code back to me so I can cancel the payment.'),
  ('c-s2', 'scam', 'dev', False, 'This is an officer from the Reserve Bank cyber cell. Your savings account has been compromised and hackers are trying to withdraw your money tonight. To protect your funds you must move your money to a safe account that I will give you. Do not tell your family or the bank staff about this call.'),
  ('c-s3', 'scam', 'dev', False, 'Hello sir, we are processing your refund for the mobile plan. To complete it please install the QuickSupport app and share your screen with me. Once the app opens, read me the code on the screen so I can start the remote session.'),
  ('c-s4', 'scam', 'dev', False, 'Your courier parcel is held at customs. To release it you need to pay the clearance fee today. I am sending a payment link, click the link and pay the amount, otherwise legal action will be taken and the parcel will be destroyed.'),
  ('c-s5', 'scam', 'dev', False, 'Congratulations, you have won the national mobile lottery. To receive your prize you only need to send the processing fee by UPI right now. Keep this confidential until the money reaches your account.'),
  ('c-b1', 'benign', 'dev', False, 'Hi, are we still meeting for lunch at one? I will bring the notes from class. Let me know if the cafe is too crowded and we can go somewhere else. See you soon.'),
  ('c-b2', 'benign', 'dev', False, 'This is a reminder from your bank. Never share your OTP, PIN or password with anyone, even if they say they are calling from the bank. If someone asks, hang up and call the number on your card.'),
  ('c-b3', 'benign', 'dev', False, 'Hello, this is the clinic. Your appointment is tomorrow at four in the afternoon. Please bring your earlier reports. If you need to move it, call us back on the clinic number.'),
  ('c-h1', 'benign', 'dev', True, 'Hello, I am the delivery executive and I am outside your building with your order. Please share the delivery code from your order app so that I can hand over the parcel.'),
  ('c-h2', 'benign', 'dev', True, 'Hello sir, this is your bank. We saw a payment on your card that looks unusual. Was it you? If not, please call the number printed on the back of your card. We will never ask for your OTP or PIN.'),
  ('c-s6', 'scam', 'test', False, "Hello, I am calling from your bank's security desk. Your card has been used for an unauthorised payment and it will be blocked today. To keep it active, tell me the verification code that just arrived on your phone."),
  ('c-s7', 'scam', 'test', False, 'This is the income tax department. There is a case against you for unpaid tax and you will be arrested tonight. To settle it immediately, transfer the amount to the account number I give you and do not speak to anyone about this.'),
  ('c-s8', 'scam', 'test', False, 'I am calling from Windows support. Your computer is sending out a virus alert and your bank details are at risk. Please download AnyDesk right now so that I can connect to your computer and remove it.'),
  ('c-s9', 'scam', 'test', False, 'Hello uncle, it is me. I am in trouble and I need money urgently, please send two thousand rupees to this number right now. Do not tell anyone in the family.'),
  ('c-s10', 'scam', 'test', False, 'Dear customer, your KYC has expired and your account will be suspended within one hour. To update it, please confirm your card number and the six digit OTP you will receive.'),
  ('c-b4', 'benign', 'test', False, 'Hi mom, I will reach home by eight. Can you keep the dinner warm? I also need to pick up the books from the shop on the way, so do not wait for me.'),
  ('c-b5', 'benign', 'test', False, 'Hey, can we move the project meeting to Thursday? The client wants to see the demo before the weekend. I will send the slides tonight so you can go through them.'),
  ('c-b6', 'benign', 'test', False, 'Hello, this is a call about your mobile plan. Your current pack ends on Sunday and a new pack with more data is available. You can recharge from the app whenever you like. Thank you.'),
  ('c-h3', 'benign', 'test', True, 'Good morning, I am calling from your bank branch. Your debit card is ready for collection. Please visit the branch with your identity proof and the receipt. We do not ask for any codes over the phone.'),
  ('c-h4', 'benign', 'test', True, 'Hey, I forgot the door code for the guest room. Can you tell me the four digit code you set last week? I am standing outside and my hands are full.'),
]

def pyt(text):
    e = tok(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        lg = m(**e)[0][0]
    return float(torch.softmax(lg, -1)[1])

print("id      label   split hard  score   verdict")
wrong = 0
for k, lab, sp, hard, txt in S:
    p = pyt(txt)
    flagged = p >= 0.5
    ok = flagged == (lab == "scam")
    wrong += (not ok)
    print("%-7s %-7s %-5s %-5s %.3f   %s" % (k, lab, sp, hard, p, "ok" if ok else "WRONG"))
print("wrong:", wrong, "of", len(S))

try:
    pos, neg = t[t.y == 1], t[t.y == 0]
    print("ICFD test: scam flagged", round(float(pos.pred.mean()), 3), "| benign flagged", round(float(neg.pred.mean()), 3))
except Exception as ex:
    print("skipped ICFD summary:", ex)

In [ ]:
print(t.groupby(["domain", "y"]).pred.agg(["mean", "count"]))
for k, lab, sp, h, txt in S:
    if lab == "scam":
        print(k, "x1 %.3f  x3 %.3f  x6 %.3f" % (pyt(txt), pyt(" ".join([txt]*3)), pyt(" ".join([txt]*6))))

In [ ]:
!cp -r /content/onnx /content/drive/MyDrive/call-shield-model-v1
!cd /content/onnx && zip -rq ../cs_model_v1.zip model_int8.onnx tokenizer.json tokenizer_config.json
!ls -lh /content/cs_model_v1.zip
from google.colab import files
files.download("/content/cs_model_v1.zip")

In [ ]:
import numpy as np, re, random
random.seed(1)
def pick2(df, per_conv, min_frac):
    df = df.copy()
    df["idx"] = np.arange(len(df))
    df["frac"] = df.chunk_timestamp / df.groupby("conversation_uid").chunk_timestamp.transform("max")
    df["y"] = (df.final_verdict == "YES").astype(int)
    keep = df[(df.y == 0) | (df.frac >= min_frac)]
    return keep.groupby("conversation_uid", group_keys=False).apply(
        lambda g: g.sample(min(per_conv, len(g)), random_state=7))
sel2 = {"train": pick2(meta["train"], 6, 0.3),
        "validation": pick2(meta["validation"], 3, 0.3),
        "test": pick2(meta["test"], 3, 0.3)}
def window(text, train):
    text = re.sub(r"\b(Agent|Customer):\s*", "", text)
    w = text.split()
    n = random.randint(30, 200) if train else 120
    return " ".join(w[-n:])
for k, v in sel2.items():
    print(k, len(v), round(float(v.y.mean()), 3))

In [ ]:
def build2(split, train):
    sub = ds_all[split].select(sel2[split].idx.tolist())
    texts = [window(x, train) for x in sub["cumulative_text"]]
    d = Dataset.from_dict({"text": texts, "label": sel2[split].y.tolist()})
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=192),
                 batched=True, remove_columns=["text"])
tr2, va2, te2 = build2("train", True), build2("validation", False), build2("test", False)
print(len(tr2), len(va2), len(te2))

In [ ]:
model2 = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)
args2 = TrainingArguments(output_dir="/content/out2", num_train_epochs=3,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=3e-5, fp16=True, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1", report_to="none")
trainer2 = Trainer(model=model2, args=args2, train_dataset=tr2, eval_dataset=va2,
    data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer2.train()

In [ ]:
m2 = trainer2.model.eval()
def pyt2(text):
    e = tok(" ".join(text.split()[-200:]), return_tensors="pt",
            truncation=True, max_length=192).to(m2.device)
    with torch.no_grad():
        return float(torch.softmax(m2(**e).logits[0], -1)[1])
wrong = 0
for k, lab, sp, hard, txt in S:
    p = pyt2(txt)
    ok = (p >= 0.5) == (lab == "scam")
    wrong += (not ok)
    print("%-7s %-7s %-5s %-5s %.3f  %s" % (k, lab, sp, hard, p, "ok" if ok else "WRONG"))
print("wrong:", wrong, "of", len(S))
pr = trainer2.predict(te2).predictions.argmax(-1)
t2 = sel2["test"].copy(); t2["pred"] = pr
print("ICFD test (120-word windows): scam flagged",
      round(float(t2[t2.y == 1].pred.mean()), 3),
      "| benign flagged", round(float(t2[t2.y == 0].pred.mean()), 3))

In [ ]:
import os, numpy as np, torch, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
m2c = m2.to("cpu").eval()
m2c.config.return_dict = False
os.makedirs("/content/onnx2", exist_ok=True)
e = tok("hello there", return_tensors="pt")
torch.onnx.export(m2c, (e["input_ids"], e["attention_mask"]), "/content/onnx2/model.onnx",
    input_names=["input_ids", "attention_mask"], output_names=["logits"],
    dynamic_axes={"input_ids": {0: "b", 1: "s"}, "attention_mask": {0: "b", 1: "s"}, "logits": {0: "b"}},
    opset_version=17, dynamo=False)
tok.save_pretrained("/content/onnx2")
quantize_dynamic("/content/onnx2/model.onnx", "/content/onnx2/model_int8.onnx", weight_type=QuantType.QInt8)
s8 = ort.InferenceSession("/content/onnx2/model_int8.onnx")
def feed(text):
    x = tok(" ".join(text.split()[-200:]), return_tensors="pt", truncation=True, max_length=192)
    return x
def p_torch(text):
    with torch.no_grad():
        lg = m2c(**feed(text))[0][0]
    return float(torch.softmax(lg, -1)[1])
def p_int8(text):
    x = feed(text)
    lg = s8.run(None, {"input_ids": x["input_ids"].numpy().astype(np.int64),
                       "attention_mask": x["attention_mask"].numpy().astype(np.int64)})[0][0]
    p = np.exp(lg - lg.max()); return float((p / p.sum())[1])
flips, worst = 0, 0.0
for k, lab, sp, hard, txt in S:
    a, b = p_torch(txt), p_int8(txt)
    worst = max(worst, abs(a - b))
    flips += ((a >= 0.5) != (b >= 0.5))
    print("%-7s torch %.3f  int8 %.3f" % (k, a, b))
print("verdict flips:", flips, "| largest difference:", round(worst, 3))
m2c.config.return_dict = True
!ls -lh /content/onnx2
!cp -r /content/onnx2 /content/drive/MyDrive/call-shield-model-v2
!cd /content/onnx2 && zip -rq ../cs_model_v2.zip model_int8.onnx tokenizer.json tokenizer_config.json
from google.colab import files
files.download("/content/cs_model_v2.zip")

In [ ]:
import numpy as np, torch, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import AutoModelForSequenceClassification

ref = {k: p_torch(txt) for k, lab, sp, h, txt in S}

quantize_dynamic("/content/onnx2/model.onnx", "/content/onnx2/model_mm8.onnx",
    weight_type=QuantType.QInt8, per_channel=True, op_types_to_quantize=["MatMul"])
sm = ort.InferenceSession("/content/onnx2/model_mm8.onnx")
def p_mm(text):
    x = feed(text)
    lg = sm.run(None, {"input_ids": x["input_ids"].numpy().astype(np.int64),
                       "attention_mask": x["attention_mask"].numpy().astype(np.int64)})[0][0]
    p = np.exp(lg - lg.max()); return float((p / p.sum())[1])

m2c.half().save_pretrained("/content/model_v2_fp16")
tok.save_pretrained("/content/model_v2_fp16")
m16 = AutoModelForSequenceClassification.from_pretrained(
    "/content/model_v2_fp16", torch_dtype=torch.float32).eval()
def p_16(text):
    with torch.no_grad():
        return float(torch.softmax(m16(**feed(text)).logits[0], -1)[1])

fl_mm = fl_16 = 0; d_mm = d_16 = 0.0
for k, lab, sp, h, txt in S:
    a, b, c = ref[k], p_mm(txt), p_16(txt)
    fl_mm += ((a >= .5) != (b >= .5)); fl_16 += ((a >= .5) != (c >= .5))
    d_mm = max(d_mm, abs(a - b)); d_16 = max(d_16, abs(a - c))
    print("%-7s real %.3f  matmul8 %.3f  fp16 %.3f" % (k, a, b, c))
print("matmul8: flips", fl_mm, "max diff", round(d_mm, 3))
print("fp16   : flips", fl_16, "max diff", round(d_16, 3))
!ls -lh /content/onnx2/model_mm8.onnx /content/model_v2_fp16
!cp -r /content/model_v2_fp16 /content/drive/MyDrive/call-shield-model-v2-fp16

In [ ]:
!cd /content && zip -rq model_v2_fp16.zip model_v2_fp16
!ls -lh /content/model_v2_fp16.zip
from google.colab import files
files.download("/content/model_v2_fp16.zip")